In [1]:
# !pip install unstructured pdf2image pillow

In [2]:
# --- Cell 1: Imports and Configuration (Must Run First) ---
import os
import base64
from pdf2image import convert_from_path
from langchain_core.documents import Document
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_community.vectorstores import Chroma

# NOTE: Since the text chunks are already processed and persisted,
# we need to load them here, but for simplicity, we'll re-run the text processing.
# You already have the successful 'chunks' variable from test.ipynb, but we'll re-run the loader.

# Configuration (Ensure Ollama is running)
CHROMA_PATH = "ollama_mrag_db"
OLLAMA_MODEL = "nomic-embed-text" # For text embedding
OLLAMA_LLM = "llava"               # For image summarization (MLLM)
DOCUMENT_PATH = "RL Intro.pdf" # Your PDF file

# Define folder for storing extracted images
IMAGE_DIR = "pdf_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

print("Imports, configuration, and image directory setup complete.")

Imports, configuration, and image directory setup complete.


In [3]:
# --- Cell 2: Image Extraction and LLaVA Summarization (Slow Step) ---

# 1. Image Extraction Helper Function
def encode_image_base64(image_path):
    """Converts a local image file to a Base64 string for the Ollama API."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 2. Extract Images (Only process the first 5 pages for speed)
try:
    print(f"Extracting images from {DOCUMENT_PATH}...")
    
    # *** PASTE YOUR FINAL POPPLER BIN PATH HERE ***
    POPPLER_BIN_PATH = r"C:\poppler-25.11.0\Library\bin" 

    # Fix: Pass the path to pdf2image to resolve the Poppler error
    images = convert_from_path(
        DOCUMENT_PATH, 
        last_page=5,
        poppler_path=POPPLER_BIN_PATH 
    ) 
    
    image_paths = []
    
    for i, img in enumerate(images):
        image_path = os.path.join(IMAGE_DIR, f"page_{i+1}.png")
        img.save(image_path, "PNG")
        image_paths.append(image_path)
    
    print(f"Extracted and saved {len(image_paths)} images in the '{IMAGE_DIR}' folder.")

except Exception as e:
    print(f"An error occurred during image extraction: {e}")
    # If this fails, the subsequent loop will result in a NameError
    image_paths = [] 

# 3. LLaVA Summarization
image_summaries = []
# Initialize LLaVA MLLM for image reasoning
llava_model = Ollama(model=OLLAMA_LLM) 

print("\nStarting LLaVA image summarization (This will take time)...")

for path in image_paths:
    base64_image = encode_image_base64(path)
    # Extract page number from the file name
    page_num = os.path.basename(path).split('_')[1].split('.')[0] 
    
    # Prompt the LLaVA model with both text and the base64-encoded image
    prompt_with_image = [
        {"role": "user", "content": [
            {"type": "text", "text": "Describe this image, chart, or diagram in detail. Summarize any key data or structural relationships shown. Be precise for the RAG system."},
            {"type": "image_url", "image_url": f"data:image/png;base64,{base64_image}"}
        ]}
    ]
    
    # Invoke the model to get the text summary
    summary = llava_model.invoke(prompt_with_image)
    
    image_summaries.append(
        {"summary": summary, "image_path": path, "source_page": page_num}
    )
    print(f"Page {page_num}: Summarized successfully.")

if image_summaries:
    print("\n--- Example Image Summary ---")
    print(f"Summary: {image_summaries[0]['summary'][:200]}...")

Extracting images from RL Intro.pdf...
Extracted and saved 5 images in the 'pdf_images' folder.

Starting LLaVA image summarization (This will take time)...


C:\Users\essal\AppData\Local\Temp\ipykernel_12020\3719354527.py:40: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llava_model = Ollama(model=OLLAMA_LLM)


Page 1: Summarized successfully.
Page 2: Summarized successfully.
Page 3: Summarized successfully.
Page 4: Summarized successfully.
Page 5: Summarized successfully.

--- Example Image Summary ---
Summary:  The image displays a color-coded chart with a grid of cells, each filled with one of three colors: green, yellow, and red. These colors are likely part of a traffic light or risk assessment system, w...


In [4]:
# --- Cell 3: Text Reload, Fusion, and Indexing ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_core.documents import Document # Ensure this is imported

# 1. Reload the original text chunks (must be done in new notebook)
loader = PyPDFLoader(DOCUMENT_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)
chunks = text_splitter.split_documents(documents)
print(f"Reloaded {len(chunks)} text chunks.")

# 2. Convert image summaries into LangChain Document format
image_documents = [
    Document(
        page_content=item['summary'],
        metadata={"source": item['image_path'], "type": "image", "page": item['source_page']}
    )
    for item in image_summaries # This variable was created in Cell 2
]

# 3. FUSE all chunks (Text + Image Summaries)
all_documents = chunks + image_documents

# 4. Re-create/Overwrite the Vector Store with the combined data
embeddings = OllamaEmbeddings(model=OLLAMA_MODEL) # nomic-embed-text
vectorstore_mrag = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

print(f"Multimodal Vector database created with {vectorstore_mrag._collection.count()} total documents.")

Reloaded 1029 text chunks.


C:\Users\essal\AppData\Local\Temp\ipykernel_12020\995779067.py:31: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model=OLLAMA_MODEL) # nomic-embed-text


Multimodal Vector database created with 2063 total documents.


In [13]:
# The chains are often still found here, but we must import from core for stability
# from langchain.chains import ConversationalRetrievalChain 

# Memory is found in the core package, NOT the main package anymore
from langchain_core.chat_history import BaseChatMessageHistory 
from langchain_core.messages import HumanMessage, AIMessage

In [16]:
!pip install langchain-classic